In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
import time
import os

In [2]:
icu_file_path = '/home/lu.kuang/thesis/Prospective_DataCleaning/updated_cleaning/DataMerged_Prospective_0430_updated.xlsx'

In [3]:
# Creating INTERVENTIONS table
INTERVENTIONS = pd.DataFrame(columns = ['ROW_ID',
                                        'INTERVENTION_ID',
                                        'ADMISSION_ID',
                                        'PATIENT_ID',
                                        'ITEM_ID',
                                        'START_DATETIME',
                                        'END_DATETIME',
                                        'VALUE_NUM',
                                        'VALUE_UOM',
                                        'VALUE_CHAR'])

In [4]:
# Creating ITEMS table
ITEMS = pd.DataFrame(columns = ['ROW_ID',
                                'ITEM_ID',
                                'LABEL',
                                'ABBREVIATION',
                                'CATEGORY',
                                'PARAM_TYPE',
                                'UOM'])

# getting reference count from measurements
count_ref_item = int(pd.read_pickle('ITEMS_MEASUREMENTS.zip')['ITEM_ID'].max()[1:]) + 1

In [5]:
# Opening ADMISSIONS for ADMISSION_ID, PATIENT_ID, RECORD_ID reference
ADMISSIONS = pd.read_csv('ADMISSIONS.csv',  delimiter = ";", encoding = 'Latin-1')
ADMISSIONS = ADMISSIONS[['ADMISSION_ID', 'PATIENT_ID', 'RECORD_ID']]
ADMISSIONS

,ADMISSION_ID,PATIENT_ID,RECORD_ID
0,A00000,P00033,189844
1,A00001,P00005,189946
2,A00002,P00033,190055
3,A00003,P00175,190072
4,A00004,P00010,190106
...,...,...,...
347,A00347,P00190,209687
348,A00348,P00076,209746
349,A00349,P00184,209792
350,A00350,P00075,209795


In [6]:
###############################################################################################################################
###############################################################################################################################

In [7]:
# preparing Dialysis for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'CRRT and IHD')
data

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,timeduration_MI,value_num,value_num_unit,OrderDoseValue_NUM,OrderDoseValue_Unit,value_char
0,190231,FMC ICU,1017255231,2021-08-08 12:16:00,Anticoagulation Type CRRT,2021-08-09 12:44:00,2021-08-09 12:44:00,0.0,NaN,NaN,NaN,NaN,Sodium Citrate
1,190231,FMC ICU,1017255231,2021-08-08 12:16:00,Comment CRRT,2021-08-09 12:44:00,2021-08-09 12:44:00,0.0,NaN,NaN,NaN,NaN,MAX 4
2,190231,FMC ICU,1017255231,2021-08-08 12:16:00,CRRT,2021-08-09 12:44:00,2021-08-10 13:32:00,1488.0,1488.00,min,1488.0,min,1488 min
3,190231,FMC ICU,1017255231,2021-08-08 12:16:00,Dialysis Catheter Right Subclavian Distal Port...,2021-08-09 12:44:00,2021-08-09 12:44:00,0.0,NaN,NaN,NaN,NaN,Aspirated for Clots
4,190231,FMC ICU,1017255231,2021-08-08 12:16:00,Dialysis Catheter Right Subclavian Distal Port...,2021-08-09 12:44:00,2021-08-09 12:44:00,0.0,NaN,NaN,NaN,NaN,Flushed with Normal Saline no Resistance Detected
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36659,206229,FMC ICU,1039962947,2022-05-03 14:33:00,Patient Ionized Calcium CRRT,2022-05-04 02:00:00,2022-05-04 02:00:00,0.0,0.96,NaN,NaN,NaN,0.96
36660,206229,FMC ICU,1039962947,2022-05-03 14:33:00,Last Hour Output CRRT,2022-05-04 02:00:00,2022-05-04 02:00:00,0.0,0.00,mL,NaN,NaN,0 mL
36661,206229,FMC ICU,1039962947,2022-05-03 14:33:00,Post Filter Ionized Calcium CRRT,2022-05-04 02:00:00,2022-05-04 02:00:00,0.0,0.27,NaN,NaN,NaN,0.27
36662,206229,FMC ICU,1039962947,2022-05-03 14:33:00,Blood Returned CRRT,2022-05-04 02:40:00,2022-05-04 02:40:00,0.0,0.00,NaN,NaN,NaN,0


In [8]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID'], axis = 1, inplace = True)

In [9]:
# Selects reference registers (CRRT and IHD)
temp = data[(data['parameter'] == 'CRRT') | (data['parameter'] == 'IHD')].copy()

In [10]:
# Adjusting overlapping interventions in temp
a = time.time()    
print("--start--")

new_temp = pd.DataFrame(columns = temp.columns)
log = pd.DataFrame(columns = ['ADMISSION_ID','chartdate', 'Case'])

# verifies available admission_ids                                     
admission_ids = temp['ADMISSION_ID'].unique()

# analysis one admission_id at a time
for admission_id in admission_ids:
    df = temp[temp['ADMISSION_ID'] == admission_id].copy()
    df.sort_values(by = ['chartdate'], inplace = True)
    df.reset_index(drop = True, inplace = True)
    df['flag'] = 0
    
    # if there is only one registered intervention, skip adjustments and save directly to new_temp
    if len(df) == 1:
        df = df[df['flag'] != 1].reset_index(drop = True)
        df.drop(['flag'], axis = 1, inplace = True)
        new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)   
        pass
    else:
        rows = range(len(df))
        for row in rows:
            if (row < rows[-1]) & (df.loc[row,'flag'] != 1):
                
                # current row
                param_t = df.loc[row, 'parameter']
                start_t = df.loc[row, 'chartdate']
                end_t = df.loc[row, 'enddate']

                # subsequent row
                param_t1 = df.loc[row + 1, 'parameter']
                start_t1 = df.loc[row + 1, 'chartdate']
                end_t1 = df.loc[row + 1, 'enddate']

                # Only condition that do not present overlapping
                if start_t1 > end_t:
                    df.loc[row,'flag'] = 0
                    df.loc[row + 1,'flag'] = 0
                else:
                    
                    # preparing support info to identify priorities
                    # verifying if complementary params support the existence of param_t (full period)
                    support_t = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                         (data['chartdate'] >= start_t) &
                                                         (data['chartdate'] <= end_t) &
                                                         (data['parameter'] != param_t)]['parameter'].unique())

                    # verifying if complementary params support the existence of param_t1 (full period)
                    support_t1 = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                           (data['chartdate'] >= start_t1) &
                                                           (data['chartdate'] <= end_t1) &
                                                           (data['parameter'] != param_t1)]['parameter'].unique())   
                    
                
                # Adjusting overlapping cases        
                # OBS: As records were sorted by chartdate, start_t1 will never be < than start_t
                
                # CASE 0
                if (start_t1 == end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,'chartdate'] = df.loc[row,'chartdate']
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 0. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)     
                    else:
                        df.loc[row,'enddate'] = df.loc[row,'enddate'] - timedelta(minutes = 1)
                        df.loc[row,'flag'] = 0
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 0. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)  
                        
                # CASE 1
                elif (start_t1 == start_t) & (end_t1 <= end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 1. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)     
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0                            
                        
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            
                            # verifying if complementary params support the existence of param_t in end section 
                            support_end = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                   (data['chartdate'] > end_t1) &
                                                                   (data['chartdate'] <= end_t) &
                                                                   (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param] 
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t_shared >= n_params_t1_shared:
                                # t is prioritized. t1 is suppressed (time_t contains time_t1)
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            elif (n_params_t_shared < n_params_t1_shared) & (support_end == True):
                                # t1 is prioritized in shared time. remaining time of t is kept
                                temp_var =  df.loc[row, 'enddate']
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'enddate']
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row + 1, 'enddate'] = temp_var
                                df.loc[row, 'parameter'] = param_t1
                                df.loc[row + 1, 'parameter'] = param_t
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                            elif (n_params_t_shared < n_params_t1_shared) & (support_end == False):                                
                                # t1 is prioritized. t is removed due the absence of complementary supporting params 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#1: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t)

                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 1. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)   
    
                # CASE 2
                elif (start_t1 == start_t) & (end_t1 > end_t):
                    if param_t == param_t1:
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 2. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                           
                    
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is removed due the absence of complementary supporting params
                            df.loc[row + 1, :] = df.loc[row, :]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is suppressed (time_t1 contains time_t)
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0   
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1                                      
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t1 in end section
                            support_end = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t) &
                                                                    (data['chartdate'] <= end_t1) &
                                                                    (data['parameter'] != param_t1)]['parameter'].unique()) 
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]   
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param] 
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t1_shared >= n_params_t_shared:
                                # t1 is prioritized. t is suppressed (time_t1 contains time_t)
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == True):
                                # t is prioritized in shared time. remaining time of t1 is kept
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate']+ timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == False):                                
                                # t is prioritized. t1 is removed due the absence of complementary supporting params 
                                df.loc[row + 1, :] = df.loc[row, :]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#2: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t) 
                                    
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 2. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)   

                # CASE 3
                elif (start_t1 < end_t) & (end_t1 == end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 3. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                          
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0  
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] >= start_t) &
                                                                    (data['chartdate'] < start_t1) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]     
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]   
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t_shared >= n_params_t1_shared:
                                # t is prioritized. t1 is suppressed (time_t contains time_t1)
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            
                            elif (n_params_t_shared < n_params_t1_shared) & (support_start == True):
                                # t1 is prioritized in shared time. remaining time of t is kept
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                                
                            elif (n_params_t_shared < n_params_t1_shared) & (support_start == False):                                
                                # t1 is prioritized. t is removed due the absence of complementary supporting params 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#3: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t) 
                                         
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 3. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)       
 
                # CASE 4
                elif (start_t1 < end_t) & (end_t1 < end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 4. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                          
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0  
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] >= start_t) &
                                                                    (data['chartdate'] < start_t1) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # verifying if complementary params support the existence of param_t in end section
                            support_end = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t1) &
                                                                    (data['chartdate'] <= end_t) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique())
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]   
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]   
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if (support_start == True) & (n_params_t1_shared >= n_params_t_shared) & (support_end == True):
                                df.loc[row + 0.5,:] = df.loc[row + 1,:]
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row, 'enddate'] = df.loc[row + 0.5, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row + 1, 'chartdate'] = df.loc[row + 0.5, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 0.5,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == True) & (n_params_t1_shared >= n_params_t_shared) & (support_end == False):
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared) & (support_end == True): 
                                temp_var =  df.loc[row, 'enddate']
                                temp_param = df.loc[row, 'parameter']
                                df.loc[row, :] = df.loc[row + 1, :]
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row + 1, 'enddate'] = temp_var
                                df.loc[row + 1, 'parameter'] = temp_param
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                            
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared) & (support_end == False): 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                            else:
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                              
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 4. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                                               
                        
                # CASE 5
                elif (start_t1 < end_t) & (end_t1 > end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,'chartdate'] = df.loc[row,'chartdate']
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 5. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                            
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is removed due the absence of complementary supporting params
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0         
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0

                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1

                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                     (data['chartdate'] >= start_t) &
                                                                     (data['chartdate'] < start_t1) &
                                                                     (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # verifying if complementary params support the existence of param_t1 in end section
                            support_end = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t) &
                                                                    (data['chartdate'] <= end_t1) &
                                                                    (data['parameter'] != param_t1)]['parameter'].unique())
                                    
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param] 
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                           
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]  
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()

                            # Prioritize the record with more occurences of a single param
                            if (support_start == True) & (n_params_t1_shared >= n_params_t_shared):
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared):
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == True): 
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                            
                            else: 
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 5. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                                     
                        
        df = df[df['flag'] != 1].reset_index(drop = True)
        df.drop(['flag'], axis = 1, inplace = True)
        new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)    
        
temp = new_temp.copy()
del new_temp
b = time.time()        
print('--end--')
print('Total processing time: %0.2f minutos' %((b-a)/60))          

--start--
--end--
Total processing time: 0.00 minutos


/tmp/ipykernel_3679201/2303343837.py:571: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)


In [11]:
# calculating delta time (after timestamps were adjusted)
def delta(x):
    return int(np.round((x['enddate'] - x['chartdate']).total_seconds()/60,0))
temp['VALUE_NUM'] = temp.apply(lambda x: delta(x), axis = 1)

In [12]:
# preparing temp to look like INTERVENTIONS table
temp.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True, ignore_index = True)
temp.drop(['value_num', 'value_char'], axis = 1, inplace = True)
temp.rename(columns={'parameter' : 'ITEM_ID',
                    'chartdate' : 'START_DATETIME',
                    'enddate' : 'END_DATETIME',
                    'value_num_unit' : 'VALUE_UOM'}, 
            inplace = True)
temp['INTERVENTION_ID'] = range(len(temp))
temp['INTERVENTION_ID'] = temp['INTERVENTION_ID'].apply(lambda x: 'IN'+str(x).rjust(8, '0'))

In [13]:
# appending to INTERVENTIONS_data
INTERVENTIONS_data = pd.concat([INTERVENTIONS, temp], axis = 0, ignore_index = True)
# Identifying the max value of INTERVENTION_ID
count_ref_intervention = int(INTERVENTIONS_data['INTERVENTION_ID'].max()[2:]) + 1
# Trying to free some memory for the following steps
del temp

/tmp/ipykernel_3679201/575765305.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  INTERVENTIONS_data = pd.concat([INTERVENTIONS, temp], axis = 0, ignore_index = True)


In [14]:
################################################################################################################################

In [15]:
# Excluding records that were already added or refer to PRESCRIPTIONS or refer to COMMENTS
temp = data[(data['parameter'] != 'CRRT') &                                            # already added
            (data['parameter'] != 'IHD') &                                             # already added
            (data['parameter'] != 'Prime IHD or MARS') &                               # PRESCRIPTIONS
            (data['parameter'] != 'Heparin for TPE or IHD') &                          # PRESCRIPTION
            (data['parameter'] != '4% Trisodium Citrate for TPE or IHD') &             # PRESCRIPTION
            (data['parameter'] != 'Hourly Comment CRRT') &                             # COMMENTS
            (data['parameter'] != 'Comment CRRT') &                                    # COMMENTS
            (data['parameter'] != 'Stopped Comment CRRT') &                            # COMMENTS
            (data['parameter'] != 'IHD Hourly Comment') &                              # COMMENTS
            (data['parameter'] != 'OLD_IHD Safety Check Comment') &                    # COMMENTS
            (data['parameter'] != 'IHD Discontinue Comment') &                         # COMMENTS
            (data['parameter'] != 'IHD Initiation Comment') &                          # COMMENTS 
            (data['parameter'] != 'Initiation Comment CRRT')].copy()                   # COMMENTS  
temp.reset_index(drop = True, inplace = True)

In [16]:
# Identifying unique combinations of ADMISSION_ID and chartdate
aux = temp.copy()
aux = aux.drop(['parameter', 'enddate', 'value_num', 'value_num_unit', 'value_char', 'PATIENT_ID'], axis = 1)
aux = aux.drop_duplicates(ignore_index = True)
aux['INTERVENTION_ID'] = np.nan

In [17]:
# Linking temp to existing INTERVENTION_ID
for n in range(len(INTERVENTIONS_data)):
    intervention_id = INTERVENTIONS_data.loc[n,'INTERVENTION_ID']
    admission_id = INTERVENTIONS_data.loc[n,'ADMISSION_ID']
    start = INTERVENTIONS_data.loc[n,'START_DATETIME']
    end = INTERVENTIONS_data.loc[n,'END_DATETIME']

    aux.loc[((aux['ADMISSION_ID'] == admission_id) & 
             (aux['chartdate'] >= start) & 
             (aux['chartdate'] <= end)), 
             'INTERVENTION_ID'] = intervention_id

/tmp/ipykernel_3679201/2455010608.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'IN00000000' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  aux.loc[((aux['ADMISSION_ID'] == admission_id) &


In [18]:
# adding INTERVENTION_ID to temp
temp = temp.merge(right = aux, on = ['ADMISSION_ID','chartdate'])
del aux

In [19]:
# preparing temp to look like INTERVENTIONS table
temp.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True, ignore_index = True)
temp.rename(columns={'parameter' : 'ITEM_ID',
                     'chartdate' : 'START_DATETIME',
                     'enddate' : 'END_DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'value_num_unit' : 'VALUE_UOM',
                     'Value_char' : 'VALUE_CHAR'}, 
            inplace = True)

In [20]:
# appending to INTERVENTIONS_data
INTERVENTIONS_data = pd.concat([INTERVENTIONS_data, temp], axis = 0, ignore_index = True)
INTERVENTIONS_data.sort_values(by = ['INTERVENTION_ID'], inplace = True, ignore_index = True)
INTERVENTIONS_data['ROW_ID'] = range(len(INTERVENTIONS_data))
del temp

In [21]:
# preparing ITEM_ID info
temp = pd.DataFrame(INTERVENTIONS_data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Dialysis'
temp['UOM'] = temp.apply(lambda x: data[data['parameter'] == x['LABEL']]['value_num_unit'].values[0], axis = 1)

def param_type(x):
    item = x['LABEL']
    num = data[data['parameter'] == item]['value_num'].values[0]
    if np.isnan(num):
        param = 'CHAR'
    else:
        param = 'NUM'
    return param    
temp['PARAM_TYPE'] = temp.apply(lambda x: param_type(x), axis = 1)
temp['LABEL'].replace({'CRRT' : 'Continuous Renal Replacement Therapy',
                       'IHD' : 'Intermittent Hemodialysis'}, inplace = True)

/tmp/ipykernel_3679201/919338158.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'CRRT' : 'Continuous Renal Replacement Therapy',


In [22]:
# Adjusting data using item_dict
INTERVENTIONS_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)

/tmp/ipykernel_3679201/1283662327.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  INTERVENTIONS_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [23]:
# appending ITEMS
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
# Identifying the max value of ITEM_ID
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

In [24]:
# saving table to pickles
INTERVENTIONS_data.to_pickle('INTERVENTIONS_01.pickle')
ITEMS_data.to_pickle('INTERVENTIONS_ITEMS_01.pickle')

# Trying to free some memory for the following steps
del INTERVENTIONS_data
del ITEMS_data
del data
del temp
del item_dict

In [25]:
################################################################################################################################
################################################################################################################################

In [26]:
# preparing INV_NIV for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'INV and NIV VENT')
data.rename(columns={'record_id':'RECORD_ID'}, inplace = True)
data.head()

,RECORD_ID,SITE,MRN,ICU_ADMIT,parameter,chartdate,enddate,timeduration_MI,value_num,value_num_unit,OrderDoseValue_NUM,OrderDoseValue_Unit,value_char
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Minute Volume High Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,0,20.0,L,NaN,NaN,20 L
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Minute Volume Low Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,0,6.0,L,NaN,NaN,6 L
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Pressure High Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,0,40.0,cmH2O,NaN,NaN,40 cmH2O
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Rate High INV,2021-08-01 02:23:00,2021-08-01 02:23:00,0,40.0,breaths/min,NaN,NaN,40 breaths/min
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Apnea Time Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,0,20.0,sec,NaN,NaN,20 sec


In [27]:
# Deleting columns that do not provide additional info (only repeat info from other columns)
data.drop(['timeduration_MI', 'OrderDoseValue_NUM', 'OrderDoseValue_Unit'], axis = 1, inplace = True)

# Excluding charvalues if num values are present
data['value_char'] = data.apply(lambda x:
                                x['value_char']
                                if np.isnan(x['value_num'])
                                else
                                np.nan,
                                axis = 1)

In [28]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID'], axis = 1, inplace = True)

In [29]:
# Identifies reference registers (HFOV, INV and NIV)
temp = data[(data['parameter'] == 'HFOV') | 
            (data['parameter'] == 'Invasive Ventilation') |
            (data['parameter'] == 'Non Invasive Ventilation')].copy()

In [30]:
# Adjusting 'parameter' 
temp['parameter'].replace({'Invasive Ventilation' : 'INV',
                          'Non Invasive Ventilation' : 'NIV'},
                          inplace = True)

/tmp/ipykernel_3679201/3188745631.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['parameter'].replace({'Invasive Ventilation' : 'INV',


In [31]:
# Adjusting overlapping interventions in temp
a = time.time()    
print("--start--")

new_temp = pd.DataFrame(columns = temp.columns)
log = pd.DataFrame(columns = ['ADMISSION_ID','chartdate', 'Case'])

# verifies available admission_ids                                     
admission_ids = temp['ADMISSION_ID'].unique()

# analysis one admission_id at a time
for admission_id in admission_ids:
    df = temp[temp['ADMISSION_ID'] == admission_id].copy()
    df.sort_values(by = ['chartdate'], inplace = True)
    df.reset_index(drop = True, inplace = True)
    df['flag'] = 0
    
    # if there is only one registered intervention, skip adjustments and save directly to new_temp
    if len(df) == 1:
        df = df[df['flag'] != 1].reset_index(drop = True)
        df.drop(['flag'], axis = 1, inplace = True)
        new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)   
        pass
    else:
        rows = range(len(df))
        for row in rows:
            if (row < rows[-1]) & (df.loc[row,'flag'] != 1):
                
                # current row
                param_t = df.loc[row, 'parameter']
                start_t = df.loc[row, 'chartdate']
                end_t = df.loc[row, 'enddate']

                # subsequent row
                param_t1 = df.loc[row + 1, 'parameter']
                start_t1 = df.loc[row + 1, 'chartdate']
                end_t1 = df.loc[row + 1, 'enddate']

                # Only condition that do not present overlapping
                if start_t1 > end_t:
                    df.loc[row,'flag'] = 0
                    df.loc[row + 1,'flag'] = 0
                else:
                    
                    # preparing support info to identify priorities
                    # verifying if complementary params support the existence of param_t (full period)
                    support_t = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                         (data['chartdate'] >= start_t) &
                                                         (data['chartdate'] <= end_t) &
                                                         (data['parameter'] != param_t)]['parameter'].unique())

                    # verifying if complementary params support the existence of param_t1 (full period)
                    support_t1 = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                           (data['chartdate'] >= start_t1) &
                                                           (data['chartdate'] <= end_t1) &
                                                           (data['parameter'] != param_t1)]['parameter'].unique())   
                    
                
                # Adjusting overlapping cases        
                # OBS: As records were sorted by chartdate, start_t1 will never be < than start_t
                
                # CASE 0
                if (start_t1 == end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,'chartdate'] = df.loc[row,'chartdate']
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 0. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)     
                    else:
                        df.loc[row,'enddate'] = df.loc[row,'enddate'] - timedelta(minutes = 1)
                        df.loc[row,'flag'] = 0
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 0. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)  
                        
                # CASE 1
                elif (start_t1 == start_t) & (end_t1 <= end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 1. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)     
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0                            
                        
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            
                            # verifying if complementary params support the existence of param_t in end section 
                            support_end = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                   (data['chartdate'] > end_t1) &
                                                                   (data['chartdate'] <= end_t) &
                                                                   (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param] 
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t_shared >= n_params_t1_shared:
                                # t is prioritized. t1 is suppressed (time_t contains time_t1)
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            elif (n_params_t_shared < n_params_t1_shared) & (support_end == True):
                                # t1 is prioritized in shared time. remaining time of t is kept
                                temp_var =  df.loc[row, 'enddate']
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'enddate']
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row + 1, 'enddate'] = temp_var
                                df.loc[row, 'parameter'] = param_t1
                                df.loc[row + 1, 'parameter'] = param_t
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                            elif (n_params_t_shared < n_params_t1_shared) & (support_end == False):                                
                                # t1 is prioritized. t is removed due the absence of complementary supporting params 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#1: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t)

                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 1. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)   
    
                # CASE 2
                elif (start_t1 == start_t) & (end_t1 > end_t):
                    if param_t == param_t1:
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 2. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                           
                    
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is removed due the absence of complementary supporting params
                            df.loc[row + 1, :] = df.loc[row, :]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is suppressed (time_t1 contains time_t)
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0   
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1                                      
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t1 in end section
                            support_end = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t) &
                                                                    (data['chartdate'] <= end_t1) &
                                                                    (data['parameter'] != param_t1)]['parameter'].unique()) 
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]   
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param] 
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t1_shared >= n_params_t_shared:
                                # t1 is prioritized. t is suppressed (time_t1 contains time_t)
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == True):
                                # t is prioritized in shared time. remaining time of t1 is kept
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate']+ timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == False):                                
                                # t is prioritized. t1 is removed due the absence of complementary supporting params 
                                df.loc[row + 1, :] = df.loc[row, :]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#2: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t) 
                                    
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 2. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)   

                # CASE 3
                elif (start_t1 < end_t) & (end_t1 == end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 3. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                          
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0  
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] >= start_t) &
                                                                    (data['chartdate'] < start_t1) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]     
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]   
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if n_params_t_shared >= n_params_t1_shared:
                                # t is prioritized. t1 is suppressed (time_t contains time_t1)
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            
                            elif (n_params_t_shared < n_params_t1_shared) & (support_start == True):
                                # t1 is prioritized in shared time. remaining time of t is kept
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0 
                                
                            elif (n_params_t_shared < n_params_t1_shared) & (support_start == False):                                
                                # t1 is prioritized. t is removed due the absence of complementary supporting params 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                            else:
                                print('ERROR#3: Unexpected condition!')
                                print('admission_id: ',admission_id)
                                print('row: ',row)
                                print('start_t: ',start_t) 
                                         
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 3. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)       
 
                # CASE 4
                elif (start_t1 < end_t) & (end_t1 < end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,:] = df.loc[row,:]
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 4. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                          
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is suppressed (time_t contains time_t1)
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0 
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0  
                            
                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1
                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] >= start_t) &
                                                                    (data['chartdate'] < start_t1) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # verifying if complementary params support the existence of param_t in end section
                            support_end = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t1) &
                                                                    (data['chartdate'] <= end_t) &
                                                                    (data['parameter'] != param_t)]['parameter'].unique())
                            
                            
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param]   
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                            
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t1) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]   
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()
                            
                            # Prioritize the record with more occurences of a single param
                            if (support_start == True) & (n_params_t1_shared >= n_params_t_shared) & (support_end == True):
                                df.loc[row + 0.5,:] = df.loc[row + 1,:]
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row, 'enddate'] = df.loc[row + 0.5, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row + 1, 'chartdate'] = df.loc[row + 0.5, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 0.5,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == True) & (n_params_t1_shared >= n_params_t_shared) & (support_end == False):
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared) & (support_end == True): 
                                temp_var =  df.loc[row, 'enddate']
                                temp_param = df.loc[row, 'parameter']
                                df.loc[row, :] = df.loc[row + 1, :]
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row + 1, 'enddate'] = temp_var
                                df.loc[row + 1, 'parameter'] = temp_param
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                            
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared) & (support_end == False): 
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                            else:
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                              
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 4. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                                               
                        
                # CASE 5
                elif (start_t1 < end_t) & (end_t1 > end_t):
                    if param_t == param_t1:
                        df.loc[row + 1,'chartdate'] = df.loc[row,'chartdate']
                        df.loc[row,'flag'] = 1
                        df.loc[row + 1,'flag'] = 0
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 5. param_t == param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                            
                    else:
                        if (support_t == True) & (support_t1 == False):
                            # t is prioritized. t1 is removed due the absence of complementary supporting params
                            df.loc[row + 1,:] = df.loc[row,:]
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0         
                            
                        elif (support_t == False) & (support_t1 == True):
                            # t1 is prioritized. t is removed due the absence of complementary supporting params
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 0

                        elif (support_t == False) & (support_t1 == False):
                            # t and t1 are removed due the absence of complementary supporting params 
                            df.loc[row,'flag'] = 1
                            df.loc[row + 1,'flag'] = 1

                        
                        # (support_t == True) & (support_t1 == True):
                        else:
                            # verifying if complementary params support the existence of param_t in start section
                            support_start = param_t in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                     (data['chartdate'] >= start_t) &
                                                                     (data['chartdate'] < start_t1) &
                                                                     (data['parameter'] != param_t)]['parameter'].unique()) 
                            
                            # verifying if complementary params support the existence of param_t1 in end section
                            support_end = param_t1 in ' '.join(data[(data['ADMISSION_ID'] == admission_id) &
                                                                    (data['chartdate'] > end_t) &
                                                                    (data['chartdate'] <= end_t1) &
                                                                    (data['parameter'] != param_t1)]['parameter'].unique())
                                    
                            # Verifying the number of occurrences of single params t in shared time 
                            params_t_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t)].copy()
                            params_t_shared = [param for param in params_t_shared_df['parameter'].unique() 
                                               if param_t in param] 
                            if len(params_t_shared) == 0:
                                n_params_t_shared = 0
                            else:
                                n_params_t_shared = params_t_shared_df[params_t_shared_df['parameter'].isin(params_t_shared)]
                                n_params_t_shared = n_params_t_shared.groupby('parameter').count()['chartdate'].max()
                           
                            
                            # Verifying the number of occurrences of single params t1 in shared time 
                            params_t1_shared_df = data[(data['ADMISSION_ID'] == admission_id) &
                                                      (data['chartdate'] >= start_t1) &
                                                      (data['chartdate'] <= end_t) &
                                                      (data['parameter'] != param_t1)].copy()
                            params_t1_shared = [param for param in params_t1_shared_df['parameter'].unique() 
                                               if param_t1 in param]  
                            if len(params_t1_shared) == 0:
                                n_params_t1_shared = 0
                            else:
                                n_params_t1_shared = params_t1_shared_df[params_t1_shared_df['parameter'].isin(params_t1_shared)]
                                n_params_t1_shared = n_params_t1_shared.groupby('parameter').count()['chartdate'].max()

                            # Prioritize the record with more occurences of a single param
                            if (support_start == True) & (n_params_t1_shared >= n_params_t_shared):
                                df.loc[row, 'enddate'] = df.loc[row + 1, 'chartdate'] - timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (support_start == False) & (n_params_t1_shared >= n_params_t_shared):
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                            elif (n_params_t1_shared < n_params_t_shared) & (support_end == True): 
                                df.loc[row + 1, 'chartdate'] = df.loc[row, 'enddate'] + timedelta(minutes = 1)
                                df.loc[row,'flag'] = 0
                                df.loc[row + 1,'flag'] = 0
                            
                            else: 
                                df.loc[row + 1,:] = df.loc[row,:]
                                df.loc[row,'flag'] = 1
                                df.loc[row + 1,'flag'] = 0
                                
                        log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id, 
                                                            'chartdate' : start_t,
                                                            'Case' : 'CASE 5. param_t != param_t1'}])], 
                                        axis = 0, 
                                        ignore_index = True)                                     
                        
        df = df[df['flag'] != 1].reset_index(drop = True)
        df.drop(['flag'], axis = 1, inplace = True)
        new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)    
        
temp = new_temp.copy()
del new_temp
b = time.time()        
print('--end--')
print('Total processing time: %0.2f minutos' %((b-a)/60))          

/tmp/ipykernel_3679201/2303343837.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_temp = pd.concat([new_temp,df], axis = 0, ignore_index = True)


--start--


/tmp/ipykernel_3679201/2303343837.py:563: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  log = pd.concat([log,pd.DataFrame([{'ADMISSION_ID' : admission_id,


--end--
Total processing time: 0.03 minutos


In [32]:
# calculating delta time (after timestamps were adjusted)
def delta(x):
    return int(np.round((x['enddate'] - x['chartdate']).total_seconds()/60,0))
temp['VALUE_NUM'] = temp.apply(lambda x: delta(x), axis = 1)

In [33]:
# preparing temp to look like INTERVENTIONS table
temp.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True, ignore_index = True)
temp.drop(['value_num', 'value_char'], axis = 1, inplace = True)
temp.rename(columns={'parameter' : 'ITEM_ID',
                    'chartdate' : 'START_DATETIME',
                    'enddate' : 'END_DATETIME',
                    'value_num_unit' : 'VALUE_UOM'}, 
            inplace = True)
temp['INTERVENTION_ID'] = range(count_ref_intervention, count_ref_intervention + len(temp))
temp['INTERVENTION_ID'] = temp['INTERVENTION_ID'].apply(lambda x: 'IN'+str(x).rjust(8, '0'))
temp['ITEM_ID'].replace({'INV' : 'Invasive Ventilation',
                         'NIV' : 'Non Invasive Ventilation'},
                        inplace = True)
temp

/tmp/ipykernel_3679201/3889333603.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['ITEM_ID'].replace({'INV' : 'Invasive Ventilation',


,SITE,MRN,ICU_ADMIT,ITEM_ID,START_DATETIME,END_DATETIME,VALUE_UOM,ADMISSION_ID,PATIENT_ID,VALUE_NUM,INTERVENTION_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Invasive Ventilation,2021-08-01 02:23:00,2021-08-02 16:57:00,min,A00000,P00033,2314,IN00000075
1,FMC ICU,1001596889,2021-08-03 03:54:00,Invasive Ventilation,2021-08-03 04:09:00,2021-08-04 12:37:00,min,A00001,P00005,1948,IN00000076
2,FMC ICU,1005018831,2021-08-04 23:56:00,Invasive Ventilation,2021-08-04 23:58:00,2021-08-06 16:00:00,min,A00002,P00033,2402,IN00000077
3,FMC ICU,1036362778,2021-08-05 06:52:00,Invasive Ventilation,2021-08-05 06:58:00,2021-08-12 11:30:00,min,A00003,P00175,10352,IN00000078
4,FMC ICU,1024147009,2021-08-05 20:48:00,Invasive Ventilation,2021-08-05 20:56:00,2021-08-24 14:19:00,min,A00005,P00125,26963,IN00000079
...,...,...,...,...,...,...,...,...,...,...,...
535,FMC ICU,1042419836,2022-07-18 16:33:00,Invasive Ventilation,2022-07-18 16:45:00,2022-07-19 12:03:00,min,A00347,P00190,1158,IN00000610
536,FMC ICU,1013229958,2022-07-19 23:23:00,Invasive Ventilation,2022-07-19 23:26:00,2022-07-21 11:44:00,min,A00348,P00076,2178,IN00000611
537,FMC ICU,1041163179,2022-07-20 22:52:00,Invasive Ventilation,2022-07-20 22:52:00,2022-07-23 01:37:00,min,A00349,P00184,3045,IN00000612
538,FMC ICU,1013129836,2022-07-21 00:22:00,Invasive Ventilation,2022-07-21 00:30:00,2022-07-21 17:29:00,min,A00350,P00075,1019,IN00000613


In [34]:
# appending to INTERVENTIONS_data
INTERVENTIONS_data = pd.concat([INTERVENTIONS, temp], axis = 0, ignore_index = True)
# Trying to free some memory for the following steps
del temp

/tmp/ipykernel_3679201/174764540.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  INTERVENTIONS_data = pd.concat([INTERVENTIONS, temp], axis = 0, ignore_index = True)


In [35]:
################################################################################################################################

In [36]:
# Excluding records that were already added or refer to COMMENTS
temp = data[(data['parameter'] != 'HFOV') &                                    # already added
            (data['parameter'] != 'Invasive Ventilation') &                    # already added
            (data['parameter'] != 'Non Invasive Ventilation') &                # already added 
            (data['parameter'] != 'Comment') &                                 # COMMENTS
            (data['parameter'] != 'Out of Range Reason Comment INV') &         # COMMENTS
            (data['parameter'] != 'Standby Comment Vent Misc') &               # COMMENTS
            (data['parameter'] != 'Comment NIV Interface') &                   # COMMENTS
            (data['parameter'] != 'Comment NIV') &                             # COMMENTS
            (data['parameter'] != 'Comment HFOV') &                            # COMMENTS
            (data['parameter'] != 'Comment V2') &                              # COMMENTS
            (data['parameter'] != 'Out of Range Reason Comment V2')].copy()    # COMMENTS
temp.reset_index(drop = True, inplace = True)

In [37]:
# Identifying unique combinations of ADMISSION_ID and chartdate
aux = temp.copy()
aux = aux.drop(['parameter', 'enddate', 'value_num', 'value_num_unit', 'value_char', 'PATIENT_ID'], axis = 1)
aux = aux.drop_duplicates(ignore_index = True)
aux['INTERVENTION_ID'] = np.nan

In [38]:
# Linking temp to existing INTERVENTION_ID
for n in range(len(INTERVENTIONS_data)):
    intervention_id = INTERVENTIONS_data.loc[n,'INTERVENTION_ID']
    admission_id = INTERVENTIONS_data.loc[n,'ADMISSION_ID']
    start = INTERVENTIONS_data.loc[n,'START_DATETIME']
    end = INTERVENTIONS_data.loc[n,'END_DATETIME']

    aux.loc[((aux['ADMISSION_ID'] == admission_id) & 
             (aux['chartdate'] >= start) & 
             (aux['chartdate'] <= end)), 
             'INTERVENTION_ID'] = intervention_id

/tmp/ipykernel_3679201/2455010608.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'IN00000075' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  aux.loc[((aux['ADMISSION_ID'] == admission_id) &


In [39]:
# adding INTERVENTION_ID to temp
temp = temp.merge(right = aux, on = ['ADMISSION_ID','chartdate'])
del aux

In [40]:
# preparing temp to look like INTERVENTIONS table
temp.sort_values(by = ['ADMISSION_ID','chartdate'], inplace = True, ignore_index = True)
temp.rename(columns={'parameter' : 'ITEM_ID',
                     'chartdate' : 'START_DATETIME',
                     'enddate' : 'END_DATETIME',
                     'value_num' : 'VALUE_NUM',
                     'value_num_unit' : 'VALUE_UOM',
                     'value_char' : 'VALUE_CHAR'}, 
            inplace = True)
temp

,SITE_x,MRN_x,ICU_ADMIT_x,ITEM_ID,START_DATETIME,END_DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,ADMISSION_ID,PATIENT_ID,SITE_y,MRN_y,ICU_ADMIT_y,INTERVENTION_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Minute Volume High Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,20.0,L,NaN,A00000,P00033,FMC ICU,1005018831,2021-08-01 02:14:00,IN00000075
1,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Minute Volume Low Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,6.0,L,NaN,A00000,P00033,FMC ICU,1005018831,2021-08-01 02:14:00,IN00000075
2,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Pressure High Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,40.0,cmH2O,NaN,A00000,P00033,FMC ICU,1005018831,2021-08-01 02:14:00,IN00000075
3,FMC ICU,1005018831,2021-08-01 02:14:00,Alarm Rate High INV,2021-08-01 02:23:00,2021-08-01 02:23:00,40.0,breaths/min,NaN,A00000,P00033,FMC ICU,1005018831,2021-08-01 02:14:00,IN00000075
4,FMC ICU,1005018831,2021-08-01 02:14:00,Apnea Time Set INV,2021-08-01 02:23:00,2021-08-01 02:23:00,20.0,sec,NaN,A00000,P00033,FMC ICU,1005018831,2021-08-01 02:14:00,IN00000075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632342,FMC ICU,1021916406,2022-07-22 21:03:00,VT Exhaled Monitored INV,2022-07-26 07:52:00,2022-07-26 07:52:00,583.8,mL,NaN,A00351,P00118,FMC ICU,1021916406,2022-07-22 21:03:00,IN00000614
632343,FMC ICU,1021916406,2022-07-22 21:03:00,VT Inspired Monitored INV,2022-07-26 07:52:00,2022-07-26 07:52:00,606.6,mL,NaN,A00351,P00118,FMC ICU,1021916406,2022-07-22 21:03:00,IN00000614
632344,FMC ICU,1021916406,2022-07-22 21:03:00,VT Set INV,2022-07-26 07:52:00,2022-07-26 07:52:00,610.0,mL,NaN,A00351,P00118,FMC ICU,1021916406,2022-07-22 21:03:00,IN00000614
632345,FMC ICU,1021916406,2022-07-22 21:03:00,Oxygen Percent_Nurse,2022-07-26 08:00:00,2022-07-26 08:00:00,100.0,%,NaN,A00351,P00118,FMC ICU,1021916406,2022-07-22 21:03:00,IN00000614


In [41]:
# appending to INTERVENTIONS_data
INTERVENTIONS_data = pd.concat([INTERVENTIONS_data, temp], axis = 0, ignore_index = True)
INTERVENTIONS_data.sort_values(by = ['INTERVENTION_ID'], inplace = True, ignore_index = True)
INTERVENTIONS_data['ROW_ID'] = range(len(INTERVENTIONS_data))
del temp

In [42]:
# preparing ITEM_ID info
temp = pd.DataFrame(INTERVENTIONS_data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Mechanical Ventilation'
temp['UOM'] = temp.apply(lambda x: data[data['parameter'] == x['LABEL']]['value_num_unit'].values[0], axis = 1)

def param_type(x):
    item = x['LABEL']
    num = data[data['parameter'] == item]['value_num'].values[0]
    if np.isnan(num):
        param = 'CHAR'
    else:
        param = 'NUM'
    return param    
temp['PARAM_TYPE'] = temp.apply(lambda x: param_type(x), axis = 1)
temp['LABEL'].replace({'HFOV' : 'High Frequency Oscillatory Ventilation'}, inplace = True)

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

/tmp/ipykernel_3679201/3927419085.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  temp['LABEL'].replace({'HFOV' : 'High Frequency Oscillatory Ventilation'}, inplace = True)


In [43]:
# Adjusting data using item_dict
INTERVENTIONS_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)

/tmp/ipykernel_3679201/1283662327.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  INTERVENTIONS_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)


In [44]:
# appending ITEMS
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
# Identifying the max value of ITEM_ID
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

In [45]:
# saving table to pickles
INTERVENTIONS_data.to_pickle('INTERVENTIONS_02.pickle')
ITEMS_data.to_pickle('INTERVENTIONS_ITEMS_02.pickle')

# Trying to free some memory for the following steps
del INTERVENTIONS_data
del ITEMS_data
del data
del temp
del item_dict

In [46]:
################################################################################################################################
################################################################################################################################
################################################################################################################################
################################################################################################################################

In [47]:
INTERVENTIONS_01 = pd.read_pickle('INTERVENTIONS_01.pickle')
INTERVENTIONS_02 = pd.read_pickle('INTERVENTIONS_02.pickle')

INTERVENTIONS = pd.concat([INTERVENTIONS_01,
                           INTERVENTIONS_02,
                          ], axis = 0, ignore_index = True)

del INTERVENTIONS_01
del INTERVENTIONS_02

INTERVENTIONS.sort_values(by = ['ADMISSION_ID','START_DATETIME'], inplace = True)
INTERVENTIONS['ROW_ID'] = range(0, len(INTERVENTIONS))
INTERVENTIONS.set_index(['ROW_ID'], inplace = True)

# INTERVENTIONS.to_csv('G:\\Projects\\ABeICU\\data\\INTERVENTIONS.csv', sep = ";", encoding = 'Latin-1')
INTERVENTIONS.to_pickle('INTERVENTIONS.zip')

os.remove('INTERVENTIONS_01.pickle')
os.remove('INTERVENTIONS_02.pickle')

In [48]:
################################################################################################################################

In [49]:
ITEMS = pd.read_pickle('ITEMS_MEASUREMENTS.pickle').reset_index()
INTERVENTIONS_ITEMS_01 = pd.read_pickle('INTERVENTIONS_ITEMS_01.pickle')
INTERVENTIONS_ITEMS_02 = pd.read_pickle('INTERVENTIONS_ITEMS_02.pickle')

INTERVENTIONS_ITEMS = pd.concat([ITEMS,
                                 INTERVENTIONS_ITEMS_01,
                                 INTERVENTIONS_ITEMS_02,
                                ], axis = 0, ignore_index = True)

del ITEMS
del INTERVENTIONS_ITEMS_01
del INTERVENTIONS_ITEMS_02

INTERVENTIONS_ITEMS.set_index(['ROW_ID'], inplace = True)

INTERVENTIONS_ITEMS.to_pickle('ITEMS_INTERVENTIONS.zip')

os.remove('INTERVENTIONS_ITEMS_01.pickle')
os.remove('INTERVENTIONS_ITEMS_02.pickle')

In [50]:
INTERVENTIONS_ITEMS

,ITEM_ID,LABEL,ABBREVIATION,CATEGORY,PARAM_TYPE,UOM
ROW_ID,,,,,,
0,I000,Eye Opening,Eye Opening,GCS,NUM,NaN
1,I001,Verbal Response,Verbal Response,GCS,NUM,NaN
2,I002,Glasgow Coma Scale,GCS,GCS,NUM,NaN
3,I003,Motor Response,Motor Response,GCS,NUM,NaN
4,I004,Verbal Response - Unable to Assess,Verbal Response - Unable to Assess,GCS,CHAR,NaN
...,...,...,...,...,...,...
319,I319,IE Ratio Monitored NIV,IE Ratio Monitored NIV,Mechanical Ventilation,CHAR,NaN
320,I320,AVAPS IPAP Minimum NIV,AVAPS IPAP Minimum NIV,Mechanical Ventilation,NUM,cmH2O
321,I321,Oxygen Flow NIV,Oxygen Flow NIV,Mechanical Ventilation,NUM,L/min


In [51]:
INTERVENTIONS

,INTERVENTION_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,START_DATETIME,END_DATETIME,VALUE_NUM,VALUE_UOM,VALUE_CHAR,SITE,...,timeduration_MI_x,OrderDoseValue_NUM_x,OrderDoseValue_Unit_x,value_char,SITE_y,MRN_y,ICU_ADMIT_y,timeduration_MI_y,OrderDoseValue_NUM_y,OrderDoseValue_Unit_y
ROW_ID,,,,,,,,,,,,,,,,,,,,,
0,IN00000075,A00000,P00033,I170,2021-08-01 02:23:00,2021-08-02 16:57:00,2314,min,NaN,FMC ICU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
1,IN00000075,A00000,P00033,I196,2021-08-01 02:23:00,2021-08-01 02:23:00,NaN,NaN,Evita XL,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,NaN,NaN,NaN
2,IN00000075,A00000,P00033,I197,2021-08-01 02:23:00,2021-08-01 02:23:00,NaN,NaN,Invasive,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,NaN,NaN,NaN
3,IN00000075,A00000,P00033,I198,2021-08-01 02:23:00,2021-08-01 02:23:00,8.0,mm,NaN,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,NaN,NaN,NaN
4,IN00000075,A00000,P00033,I199,2021-08-01 02:23:00,2021-08-01 02:23:00,100.0,%,NaN,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.005019e+09,2021-08-01 02:14:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
669784,IN00000614,A00351,P00118,I183,2022-07-26 07:52:00,2022-07-26 07:52:00,NaN,NaN,VC357475,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,NaN,NaN,NaN
669785,IN00000614,A00351,P00118,I208,2022-07-26 07:52:00,2022-07-26 07:52:00,6502.0,mL,NaN,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,NaN,NaN,NaN
669786,IN00000614,A00351,P00118,I171,2022-07-26 07:52:00,2022-07-26 07:52:00,18.0,cmH2O,NaN,NaN,...,NaN,NaN,NaN,NaN,FMC ICU,1.021916e+09,2022-07-22 21:03:00,NaN,NaN,NaN


In [52]:
INTERVENTIONS.to_csv('INTERVENTIONS.csv', sep = ";", encoding = 'Latin-1')

In [53]:
INTERVENTIONS_ITEMS.to_csv('INTERVENTIONS_ITEMS.csv', sep = ";", encoding = 'Latin-1')